<a href="https://colab.research.google.com/github/edizan12/Synthetic-Intelligence-Architecture/blob/main/si_engine_colab_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SI Engine v9.5.4 — Cross-Tokenizer Contrastive Decoding
Expert (Llama) - alpha * Amateur (Qwen), vocabulary mapping + subtoken proxy synchronization.

This notebook is prepared to run in Google Colab (and can be tested on a free-tier T4 GPU).
Run the cells in order.

## 1. Install dependencies

In [ ]:
!pip install -q -U transformers accelerate bitsandbytes

## 2. (Optional) Hugging Face login
`meta-llama/Llama-3.2-3B-Instruct` is a gated model. If you have access, run the cell below
and enter your token. Alternatively, you can use a non-gated model such as
`Qwen/Qwen2.5-3B-Instruct` instead of the expert model.

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

## 3. Imports and configuration

In [ ]:
import os
import sys
import time
import math
from collections import Counter
import torch
import torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

EXPERT_MODEL = "meta-llama/Llama-3.2-3B-Instruct"
AMATEUR_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
USE_4BIT = True
SEED = 42

torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Active device: {DEVICE.upper()}")

## 4. Load models

In [ ]:
def load_models(expert_name, amateur_name, device, use_4bit=True):
    qc = None
    if use_4bit and device == "cuda":
        qc = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_quant_type="nf4",
        )
    elif use_4bit and device != "cuda":
        print("WARNING: 4-bit quantization requires CUDA; loading full precision instead.")

    print("\nLoading tokenizers and model weights...")
    tok_e = AutoTokenizer.from_pretrained(expert_name)
    tok_a = AutoTokenizer.from_pretrained(amateur_name)

    kw = {}
    if qc is not None:
        kw["quantization_config"] = qc
        kw["device_map"] = "auto"
    else:
        kw["torch_dtype"] = torch.float16 if device == "cuda" else torch.float32

    model_e = AutoModelForCausalLM.from_pretrained(expert_name, **kw)
    model_a = AutoModelForCausalLM.from_pretrained(amateur_name, **kw)

    if qc is None:
        model_e = model_e.to(device)
        model_a = model_a.to(device)

    model_e.eval()
    model_a.eval()
    return tok_e, tok_a, model_e, model_a

tok_e, tok_a, model_e, model_a = load_models(EXPERT_MODEL, AMATEUR_MODEL, DEVICE, use_4bit=USE_4BIT)

## 5. Vocabulary mapping (identity shortcut + explicit mapping + vectorized proxy)

In [ ]:
def build_vocab_map(tok_e, tok_a, model_e, model_a, device):
    """
    Returns:
        mapping        : (V_e,) long, -1 = no match
        subtoken_map   : dict eid -> List[amateur_subtoken_ids]
        proxy_ids_t    : (P,) long tensor
        proxy_matrix   : (P, L) long tensor  (pad = 0, positional mask applied)
        proxy_lengths  : (P,) long tensor
        proxy_valid    : (P, L) bool tensor  (padding positions are False)
    """
    print("\nBuilding vocabulary mapping (v4)...")

    V_e = model_e.get_output_embeddings().weight.shape[0]
    V_a = model_a.get_output_embeddings().weight.shape[0]

    if tok_e.get_vocab() == tok_a.get_vocab() and V_e == V_a:
        print("Same tokenizer + same embedding size -> identity mapping.")
        ids = torch.arange(V_e, dtype=torch.long, device=device)
        empty_ids = torch.empty(0, dtype=torch.long, device=device)
        empty_mat = torch.empty(0, 0, dtype=torch.long, device=device)
        empty_len = torch.empty(0, dtype=torch.long, device=device)
        empty_val = torch.empty(0, 0, dtype=torch.bool, device=device)
        return ids, {}, empty_ids, empty_mat, empty_len, empty_val

    if tok_e.get_vocab() == tok_a.get_vocab() and V_e != V_a:
        print(f"WARNING: same tokenizer but different embedding sizes "
              f"(expert={V_e}, amateur={V_a}). Falling back to explicit mapping.")

    mapping = torch.full((V_e,), -1, dtype=torch.long, device=device)
    subtoken_map = {}

    for eid in range(V_e):
        tok_str = tok_e.convert_ids_to_tokens(eid)
        if tok_str is None:
            continue
        if tok_str.startswith("<") and tok_str.endswith(">"):
            continue
        try:
            word = tok_e.convert_tokens_to_string([tok_str])
        except Exception:
            continue
        if not word:
            continue
        am_ids = tok_a.encode(word, add_special_tokens=False)
        if len(am_ids) == 1 and am_ids[0] < V_a:
            mapping[eid] = am_ids[0]
        elif len(am_ids) > 1 and all(a < V_a for a in am_ids):
            subtoken_map[eid] = am_ids

    mapped = (mapping >= 0).sum().item()
    proxy = len(subtoken_map)
    print(f"1-to-1: {mapped}, proxy: {proxy}, "
          f"covered: {mapped + proxy}/{V_e} "
          f"({100 * (mapped + proxy) / V_e:.1f}%)")

    proxy_ids = list(subtoken_map.keys())
    if proxy_ids:
        max_len = max(len(subtoken_map[e]) for e in proxy_ids)
        proxy_matrix = torch.zeros((len(proxy_ids), max_len), dtype=torch.long)
        proxy_lengths = torch.zeros(len(proxy_ids), dtype=torch.long)
        for i, e in enumerate(proxy_ids):
            subs = subtoken_map[e]
            proxy_matrix[i, :len(subs)] = torch.tensor(subs, dtype=torch.long)
            proxy_lengths[i] = len(subs)

        proxy_ids_t = torch.tensor(proxy_ids, dtype=torch.long, device=device)
        proxy_matrix = proxy_matrix.to(device)
        proxy_lengths = proxy_lengths.to(device)

        arange = torch.arange(max_len, device=device).unsqueeze(0)
        proxy_valid = arange < proxy_lengths.unsqueeze(1)
    else:
        proxy_ids_t = torch.empty(0, dtype=torch.long, device=device)
        proxy_matrix = torch.empty(0, 0, dtype=torch.long, device=device)
        proxy_lengths = torch.empty(0, dtype=torch.long, device=device)
        proxy_valid = torch.empty(0, 0, dtype=torch.bool, device=device)

    return mapping, subtoken_map, proxy_ids_t, proxy_matrix, proxy_lengths, proxy_valid

(expert_to_amateur_map, subtoken_map,
 proxy_ids_t, proxy_mat, proxy_len, proxy_val) = build_vocab_map(tok_e, tok_a, model_e, model_a, DEVICE)

## 6. Warm-up

In [ ]:
def warm_up(model_e, model_a, tok_e, tok_a, device):
    print(f"\nWarming up {device.upper()} kernels...")
    d_e = torch.tensor([[tok_e.eos_token_id]], device=device)
    d_a = torch.tensor([[tok_a.eos_token_id]], device=device)
    with torch.no_grad():
        _ = model_e(d_e, attention_mask=torch.ones_like(d_e), use_cache=False)
        _ = model_a(d_a, attention_mask=torch.ones_like(d_a), use_cache=False)

warm_up(model_e, model_a, tok_e, tok_a, DEVICE)

## 7. Generation function (fusion + adaptive plausibility + repetition penalty)

In [ ]:
@torch.no_grad()
def generate(user_query, tok_e, tok_a, model_e, model_a, expert_to_amateur_map,
             proxy_ids_t, proxy_matrix, proxy_lengths, proxy_valid, device,
             alpha=0.40, repetition_penalty=0.30, penalty_window=20,
             beta=0.10, temperature=0.0, top_p=1.0,
             soft_limit=120, absolute_max=200, verbose=True):

    if beta > 0:
        beta = min(max(beta, 1e-8), 1.0)

    msg_e = tok_e.apply_chat_template(
        [{"role": "user", "content": user_query}],
        tokenize=False, add_generation_prompt=True)
    msg_a = tok_a.apply_chat_template(
        [{"role": "user", "content": user_query}],
        tokenize=False, add_generation_prompt=True)

    input_e = tok_e([msg_e], return_tensors="pt").input_ids.to(device)
    input_a = tok_a([msg_a], return_tensors="pt").input_ids.to(device)

    past_e = past_a = None
    mask_e = torch.ones_like(input_e)
    mask_a = torch.ones_like(input_a)
    curr_e, curr_a = input_e, input_a

    if device == "cuda":
        torch.cuda.reset_peak_memory_stats()

    start = time.time()
    total = 0
    gen_ids = []
    out_parts = []

    V = expert_to_amateur_map.shape[0]
    valid_mask = expert_to_amateur_map >= 0
    mapped_am = torch.zeros(1, V, device=device)

    for step in range(absolute_max):
        out_e = model_e(input_ids=curr_e, past_key_values=past_e,
                        attention_mask=mask_e, use_cache=True)
        logits_e = out_e.logits[:, -1, :]
        past_e = out_e.past_key_values

        out_a = model_a(input_ids=curr_a, past_key_values=past_a,
                        attention_mask=mask_a, use_cache=True)
        logits_a = out_a.logits[:, -1, :]
        past_a = out_a.past_key_values

        # NOTE: the expert and amateur model compute dtypes (fp16/bf16/4-bit quantization)
        # may differ. We cast immediately after log-softmax so fusion/threshold/index operations
        # are always performed in float32.
        lp_e = F.log_softmax(logits_e, dim=-1).float()
        lp_a = F.log_softmax(logits_a, dim=-1).float()

        mapped_am.zero_()
        mapped_am[0, valid_mask] = lp_a[0, expert_to_amateur_map[valid_mask]]

        if proxy_ids_t.numel() > 0:
            sub_lps = lp_a[0, proxy_matrix].masked_fill(~proxy_valid, 0.0)
            proxy_means = sub_lps.sum(-1) / proxy_lengths.float()
            mapped_am[0, proxy_ids_t] = proxy_means

        fused = lp_e - alpha * mapped_am

        if beta > 0:
            threshold = lp_e.max(dim=-1, keepdim=True).values + math.log(beta)
            fused = fused.masked_fill(lp_e < threshold, float("-inf"))

        if gen_ids:
            counts = Counter(gen_ids[-penalty_window:])
            for tid, c in counts.items():
                fused[0, tid] -= repetition_penalty * c

        if step >= absolute_max - 3:
            fused[0, tok_e.eos_token_id] = fused[0].max() + 1.0

        if temperature > 0:
            probs = F.softmax(fused / temperature, dim=-1)
            if top_p < 1.0:
                sorted_probs, sorted_idx = torch.sort(probs, descending=True, dim=-1)
                cumsum = sorted_probs.cumsum(-1)
                mask = cumsum - sorted_probs > top_p
                sorted_probs[mask] = 0
                sorted_probs /= sorted_probs.sum(-1, keepdim=True)
                pick = torch.multinomial(sorted_probs, 1)
                nid = sorted_idx[0, pick].item()
            else:
                nid = torch.multinomial(probs, 1).item()
        else:
            nid = torch.argmax(fused, dim=-1).item()

        if nid == tok_e.eos_token_id:
            break

        total += 1
        gen_ids.append(nid)
        word = tok_e.decode([nid])
        if verbose:
            print(word, end="", flush=True)
        out_parts.append(word)

        if step >= soft_limit:
            cum = "".join(out_parts).rstrip()
            if any(cum.endswith(p) for p in [".", "!", "?"]):
                break

        curr_e = torch.tensor([[nid]], device=device)
        mask_e = torch.cat(
            [mask_e, torch.ones((1, 1), device=device, dtype=mask_e.dtype)], dim=-1)

        a_id = expert_to_amateur_map[nid].item()
        if a_id >= 0:
            curr_a = torch.tensor([[a_id]], device=device)
            mask_a = torch.cat(
                [mask_a, torch.ones((1, 1), device=device, dtype=mask_a.dtype)], dim=-1)
        else:
            subs = tok_a.encode(word, add_special_tokens=False)
            if not subs:
                continue
            if len(subs) > 1:
                bi = torch.tensor([subs[:-1]], device=device)
                bm = torch.cat(
                    [mask_a,
                     torch.ones((1, bi.shape[-1]), device=device, dtype=mask_a.dtype)],
                    dim=-1)
                out_a2 = model_a(input_ids=bi, past_key_values=past_a,
                                 attention_mask=bm, use_cache=True)
                past_a = out_a2.past_key_values
                mask_a = bm
            curr_a = torch.tensor([[subs[-1]]], device=device)
            mask_a = torch.cat(
                [mask_a, torch.ones((1, 1), device=device, dtype=mask_a.dtype)], dim=-1)

    dur = time.time() - start
    tps = total / dur if dur > 0 else 0
    vram = torch.cuda.max_memory_allocated() / 1e9 if device == "cuda" else 0.0

    print("\n" + "-" * 60)
    print(f"alpha={alpha}, beta={beta}, rep={repetition_penalty}, T={temperature}")
    print(f"{tps:.2f} tok/s | {vram:.2f} GB | {total} tokens")
    print("-" * 60)
    return "".join(out_parts)

## 8. Run
In Colab, the most practical approach is to change the `PROMPT` variable directly and rerun the cell
instead of waiting for `input()`.

In [ ]:
PROMPT = "Briefly explain quantum entanglement."

_ = generate(
    PROMPT, tok_e, tok_a, model_e, model_a, expert_to_amateur_map,
    proxy_ids_t, proxy_mat, proxy_len, proxy_val, DEVICE,
    alpha=0.40, repetition_penalty=0.30, penalty_window=20,
    beta=0.10, temperature=0.0, top_p=1.0,
    soft_limit=120, absolute_max=200,
)

## 9. (Optional) Interactive loop
`input()` works in Colab, but it pauses the cell and waits for input at every step; for longer sessions,
prefer the single-shot cell above. You can still use the loop below if needed.

In [ ]:
while True:
    user_query = input("\nUser prompt (type exit/quit to quit): ").strip()
    if user_query.lower() in ("exit", "quit"):
        print("SI Engine: Session terminated.")
        break
    if not user_query:
        continue
    print("\nSI Output: ", end="", flush=True)
    generate(
        user_query, tok_e, tok_a, model_e, model_a, expert_to_amateur_map,
        proxy_ids_t, proxy_mat, proxy_len, proxy_val, DEVICE,
        alpha=0.40, repetition_penalty=0.30, penalty_window=20,
        beta=0.10, temperature=0.0, top_p=1.0,
        soft_limit=120, absolute_max=200,
    )